# 077 — LoRA, QLoRA y adaptación eficiente

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** A: 8×768 = 6 144; B: 768×8 = 6 144; total LoRA = **12 288**.
Matriz completa: 768² = **589 824**. Porcentaje: 12 288/589 824 ≈ **2,08 %**.
Con r = 64: 64·(768+768) = 98 304 → 16,7 %: sigue siendo parcial, pero el ahorro
se erosiona — r es un dial de capacidad/costo.

**Ejercicio 2.** r·(d+k) = 8·(768+3072) = **30 720** frente a 768·3072 = 2 359 296:
**1,30 %**. La fracción es *menor* que en el caso cuadrado porque d·k crece más
rápido que d+k cuando las dimensiones crecen.

**Ejercicio 3.** (a) 7e9·2 B = **14 GB**. (b) 7e9·0,55 B ≈ **3,85 GB** (~3,6×
menos). Adam sobre 20 M LoRA: 20e6·12 = **0,24 GB** — el optimizador deja de ser el
problema; por eso QLoRA cabe en una GPU de escritorio.

**Ejercicio 4.** El laboratorio entrena/ajusta un núcleo mínimo: su limitación
declara que la evidencia no equivale a validación de producción — exactamente la
brecha entre costo de entrenar y costo de evaluar.

In [ ]:
# Ejercicio 1
d = k = 768; r = 8
params_lora = r * (d + k)
params_full = d * k
print(params_lora, params_full, f"{params_lora/params_full:.2%}")  # 12288 589824 2.08%
r = 64
print(64 * (768 + 768), f"{64*(768+768)/params_full:.2%}")  # 98304 16.67%

# Ejercicio 2
d2, k2 = 768, 3072
print(8 * (d2 + k2), d2 * k2, f"{8*(d2+k2)/(d2*k2):.2%}")  # 30720 2359296 1.30%

# Ejercicio 3
N = 7e9
print(f"FP16: {N*2/1e9:.1f} GB | NF4: {N*0.55/1e9:.2f} GB | Adam LoRA: {20e6*12/1e9:.2f} GB")

# Ejercicio 4
result = run_lab("neural", seed=77)
assert result["kind"] == "neural"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué B se inicializa en cero y A gaussiana, y qué pasaría al inicio del
   entrenamiento si ambas fueran gaussianas?
2. Parámetros LoRA crecen como r·(d+k) y la matriz completa como d·k: ¿por qué esto
   hace a LoRA *relativamente* más eficiente cuanto más grande es el modelo?
3. ¿En qué escenario de producto elegirías servir 20 adaptadores sin fusionar sobre
   un base compartido en vez de 20 modelos fusionados, y qué pagas a cambio?